# Autoencoder for Anomaly Detection on CICIDS2018 (Google Colab)

This notebook trains an Autoencoder for anomaly detection on the CICIDS2018 dataset, optimised to run on **Google Colab with GPU acceleration**.

**Model:** Autoencoder (PyTorch)  
**Dataset:** CICIDS2018  
**Task:** Anomaly Detection (Benign vs Attack based on reconstruction error)

## Setup Instructions
1. **Enable GPU**: `Runtime → Change runtime type → T4 GPU`
2. **Upload data to Google Drive** at `MyDrive/NIDS-DL/data/raw/cicids2018/` (CSV files)
3. **Run cells top-to-bottom**

## Anomaly Detection Approach
1. Train autoencoder exclusively on **benign (normal) traffic**
2. The model learns to reconstruct normal traffic with low error
3. Attack traffic produces **high reconstruction error** (anomaly)
4. Set a threshold on reconstruction error to classify samples

## 0. GPU Check & Google Drive Mount

In [ ]:
# Check GPU availability — go to Runtime > Change runtime type > T4 GPU if not available
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ GPU is available!")
    print(result.stdout[:500])
else:
    print("❌ No GPU detected.")
    print("Go to Runtime → Change runtime type → select 'T4 GPU' → Save")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("Google Drive mounted at /content/drive")

## 1. Setup and Imports

In [ ]:
import sys
import os
import glob
import time
import gc
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
import seaborn as sns

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  Running on CPU — training will be much slower.")
    print("    Go to Runtime → Change runtime type → T4 GPU")

## 2. Configuration

In [ ]:
# =============================================================================
# ⚙️  CONFIGURATION — Adjust these paths/parameters as needed
# =============================================================================

# --- Data paths (Google Drive) ---
# Upload CICIDS2018 CSV files to this folder in your Google Drive:
DATA_DIR = '/content/drive/MyDrive/NIDS-DL/data/raw/cicids2018/'

# --- Results/model output paths ---
RESULTS_DIR = '/content/drive/MyDrive/NIDS-DL/results/'
MODEL_SAVE_PATH = os.path.join(RESULTS_DIR, 'models', 'best_autoencoder_cicids2018.pth')
RESULTS_JSON_PATH = os.path.join(RESULTS_DIR, 'autoencoder_cicids2018_results.json')

# --- Data sampling (reduce if RAM is limited) ---
NORMAL_SAMPLES  = 500_000   # Max benign samples to use
ATTACK_SAMPLES  = 200_000   # Max attack samples to use
MAX_FILE_SIZE_MB = 1000     # Skip CSV files larger than this

# --- Model hyperparameters ---
ENCODING_DIM = 16           # Bottleneck dimension
DROPOUT_RATE = 0.2

# --- Training hyperparameters ---
BATCH_SIZE = 2048           # Larger batches for GPU efficiency
EPOCHS     = 30
LR         = 0.001
EARLY_STOPPING_PATIENCE = 7

# Create output directories
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
os.makedirs(os.path.dirname(RESULTS_JSON_PATH), exist_ok=True)

print("Configuration:")
print(f"  Data dir:       {DATA_DIR}")
print(f"  Model save:     {MODEL_SAVE_PATH}")
print(f"  Normal samples: {NORMAL_SAMPLES:,}")
print(f"  Attack samples: {ATTACK_SAMPLES:,}")
print(f"  Batch size:     {BATCH_SIZE}")
print(f"  Epochs:         {EPOCHS}")
print(f"  Encoding dim:   {ENCODING_DIM}")

## 3. Load and Preprocess Data

In [ ]:
# Discover CSV files
all_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))

if not all_files:
    raise FileNotFoundError(
        f"No CSV files found in {DATA_DIR}\n"
        "Please upload CICIDS2018 CSV files to Google Drive at:\n"
        f"  {DATA_DIR}"
    )

print(f"Found {len(all_files)} CSV file(s).")
li = []

for filename in all_files:
    file_size_mb = os.path.getsize(filename) / (1024 * 1024)
    if file_size_mb > MAX_FILE_SIZE_MB:
        print(f"  ⏭ Skipping {os.path.basename(filename)} ({file_size_mb:.0f} MB — too large)")
        continue
    print(f"  📂 Loading {os.path.basename(filename)} ({file_size_mb:.1f} MB)...")
    try:
        df_temp = pd.read_csv(filename, index_col=None, header=0, low_memory=False)
        li.append(df_temp)
    except Exception as e:
        print(f"  ❌ Error loading {filename}: {e}")

if not li:
    raise RuntimeError("No files were loaded successfully.")

df = pd.concat(li, axis=0, ignore_index=True)
print(f"\nTotal raw samples: {len(df):,}")
del li
gc.collect()

In [ ]:
# Preprocessing

# 1. Strip whitespace from column names
df.columns = df.columns.str.strip()

# 2. Drop metadata columns
drop_cols = ['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp']
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True, errors='ignore')

# 3. Convert feature columns to numeric
feature_cols = [c for c in df.columns if c != 'Label']
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 4. Replace Inf/NaN and drop
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

print(f"Samples after cleaning: {len(df):,}")
print(f"Feature columns:        {len(feature_cols)}")
print("\nLabel distribution:")
print(df['Label'].value_counts().to_string())

In [ ]:
# Create Binary Label: Benign (0) vs Attack (1)
def create_binary_label(label):
    if isinstance(label, str) and 'BENIGN' in label.upper():
        return 0
    return 1

df['binary_label'] = df['Label'].apply(create_binary_label)

n_benign = (df['binary_label'] == 0).sum()
n_attack = (df['binary_label'] == 1).sum()
print(f"Benign samples: {n_benign:,}  ({n_benign/len(df)*100:.1f}%)")
print(f"Attack samples: {n_attack:,}  ({n_attack/len(df)*100:.1f}%)")

# Separate features and labels
y = df['binary_label'].values
X = df.drop(columns=['Label', 'binary_label']).values.astype(np.float32)

print(f"\nFeature matrix shape: {X.shape}")

del df
gc.collect()

In [ ]:
# Sample data (important for memory management on Colab)
normal_idx = np.where(y == 0)[0]
attack_idx = np.where(y == 1)[0]

np.random.seed(SEED)
if len(normal_idx) > NORMAL_SAMPLES:
    normal_idx = np.random.choice(normal_idx, NORMAL_SAMPLES, replace=False)
if len(attack_idx) > ATTACK_SAMPLES:
    attack_idx = np.random.choice(attack_idx, ATTACK_SAMPLES, replace=False)

X_normal = X[normal_idx]
X_attack = X[attack_idx]
y_attack  = np.ones(len(X_attack), dtype=np.int64)

print(f"Benign samples selected: {len(X_normal):,}")
print(f"Attack samples selected: {len(X_attack):,}")

del X, y
gc.collect()

In [ ]:
# Train/Val/Test split
# For autoencoder: TRAIN on benign-only data

X_train, X_val_full = train_test_split(X_normal, test_size=0.3, random_state=SEED)
X_val, X_test_normal = train_test_split(X_val_full, test_size=0.5, random_state=SEED)
del X_val_full

# Mixed test set (benign + attacks)
X_test = np.vstack([X_test_normal, X_attack])
y_test = np.hstack([
    np.zeros(len(X_test_normal), dtype=np.int64),
    y_attack
])

print(f"Train samples (benign only): {X_train.shape[0]:,}")
print(f"Val samples   (benign only): {X_val.shape[0]:,}")
print(f"Test samples  (mixed):       {X_test.shape[0]:,}")
print(f"  ↳ Benign:                  {np.sum(y_test == 0):,}")
print(f"  ↳ Attack:                  {np.sum(y_test == 1):,}")

In [ ]:
# Feature scaling (StandardScaler fit on training data only)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val   = scaler.transform(X_val).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

print("Feature scaling complete.")
print(f"X_train stats — mean: {X_train.mean():.4f}, std: {X_train.std():.4f}")

In [ ]:
# Create DataLoaders
# pin_memory=True speeds up CPU→GPU transfers
def make_loader(X, shuffle=True, batch_size=BATCH_SIZE):
    X_t = torch.FloatTensor(X)
    dataset = TensorDataset(X_t, X_t)  # Input = Target for reconstruction
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

train_loader = make_loader(X_train, shuffle=True)
val_loader   = make_loader(X_val,   shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Batch size:    {BATCH_SIZE}")

## 4. Define Autoencoder Model

In [ ]:
class Autoencoder(nn.Module):
    """
    Deep Autoencoder for network anomaly detection.

    Architecture:
      Encoder: input_dim → 128 → 64 → 32 → encoding_dim
      Decoder: encoding_dim → 32 → 64 → 128 → input_dim

    Uses BatchNorm + Dropout for regularisation.
    Anomaly detection via reconstruction error threshold.
    """
    def __init__(self, input_dim, encoding_dim=16, dropout_rate=0.2):
        super(Autoencoder, self).__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Linear(32, encoding_dim),
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Linear(128, input_dim),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


input_dim = X_train.shape[1]
model = Autoencoder(
    input_dim=input_dim,
    encoding_dim=ENCODING_DIM,
    dropout_rate=DROPOUT_RATE
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nInput dimension:  {input_dim}")
print(f"Encoding dim:     {ENCODING_DIM}")
print(f"Total parameters: {total_params:,}")

## 5. Training

In [ ]:
criterion  = nn.MSELoss()
optimizer  = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler  = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5, verbose=True
)

print(f"Optimizer: Adam (lr={LR}, weight_decay=1e-5)")
print(f"Scheduler: ReduceLROnPlateau (patience=3, factor=0.5)")
print(f"Epochs: {EPOCHS}, Early Stopping patience: {EARLY_STOPPING_PATIENCE}")

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    for X_batch, _ in loader:
        X_batch = X_batch.to(device, non_blocking=True)
        optimizer.zero_grad()
        recon = model(X_batch)
        loss  = criterion(recon, X_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    return total_loss / len(loader.dataset)


def validate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for X_batch, _ in loader:
            X_batch = X_batch.to(device, non_blocking=True)
            recon   = model(X_batch)
            loss    = criterion(recon, X_batch)
            total_loss += loss.item() * X_batch.size(0)
    return total_loss / len(loader.dataset)

In [ ]:
best_val_loss    = float('inf')
patience_counter = 0
train_losses     = []
val_losses       = []

print("Starting training...")
print("=" * 70)

for epoch in range(EPOCHS):
    t0 = time.time()

    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss   = validate(model, val_loader, criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    elapsed = time.time() - t0
    print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
          f"Train: {train_loss:.6f} | "
          f"Val: {val_loss:.6f} | "
          f"Time: {elapsed:.1f}s", end="")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        patience_counter = 0
        print(" ✅ saved")
    else:
        patience_counter += 1
        print(f" (patience {patience_counter}/{EARLY_STOPPING_PATIENCE})")
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print("\nEarly stopping triggered.")
            break

print("=" * 70)
print(f"Training complete. Best val loss: {best_val_loss:.6f}")
print(f"Model saved to: {MODEL_SAVE_PATH}")

In [ ]:
# Plot training history
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', linewidth=2, color='steelblue')
plt.plot(val_losses,   label='Val Loss',   linewidth=2, color='darkorange')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Autoencoder Training History — CICIDS2018')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Anomaly Detection Evaluation

In [ ]:
# Load best model weights
model.load_state_dict(torch.load(MODEL_SAVE_PATH, weights_only=True))
model.eval()

# Compute reconstruction errors on test set
reconstruction_errors = []
INFER_BATCH = 4096

with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test)
    for i in range(0, len(X_test_tensor), INFER_BATCH):
        batch = X_test_tensor[i:i+INFER_BATCH].to(device, non_blocking=True)
        recon = model(batch)
        mse   = torch.mean((batch - recon) ** 2, dim=1)
        reconstruction_errors.extend(mse.cpu().numpy())

reconstruction_errors = np.array(reconstruction_errors)
print(f"Reconstruction errors — shape: {reconstruction_errors.shape}")
print(f"  Benign  — mean: {reconstruction_errors[y_test==0].mean():.6f}")
print(f"  Attack  — mean: {reconstruction_errors[y_test==1].mean():.6f}")

In [ ]:
# Reconstruction error distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(reconstruction_errors[y_test == 0], bins=100, alpha=0.7,
             label='Benign', color='steelblue', density=True)
axes[0].hist(reconstruction_errors[y_test == 1], bins=100, alpha=0.7,
             label='Attack', color='crimson', density=True)
axes[0].set_xlabel('Reconstruction Error (MSE)')
axes[0].set_ylabel('Density')
axes[0].set_title('Reconstruction Error Distribution')
axes[0].legend()
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

axes[1].boxplot(
    [reconstruction_errors[y_test == 0], reconstruction_errors[y_test == 1]],
    tick_labels=['Benign', 'Attack'],
    showfliers=False
)
axes[1].set_ylabel('Reconstruction Error (MSE)')
axes[1].set_title('Reconstruction Error by Class')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ROC curve & optimal threshold (Youden's J statistic)
fpr, tpr, thresholds = roc_curve(y_test, reconstruction_errors)
roc_auc = auc(fpr, tpr)

j_scores   = tpr - fpr
opt_idx    = np.argmax(j_scores)
opt_thresh = thresholds[opt_idx]

print(f"ROC AUC:              {roc_auc:.4f}")
print(f"Optimal threshold:    {opt_thresh:.6f}")
print(f"At optimal threshold: TPR={tpr[opt_idx]:.4f}, FPR={fpr[opt_idx]:.4f}")

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
plt.scatter(fpr[opt_idx], tpr[opt_idx], color='red', s=120, zorder=5,
            label=f'Optimal threshold: {opt_thresh:.4f}')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Autoencoder CICIDS2018')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Threshold sweep — evaluate at different percentiles on validation set

# Get reconstruction errors on clean val set (benign only)
val_errors = []
with torch.no_grad():
    for i in range(0, len(X_val), INFER_BATCH):
        batch = torch.FloatTensor(X_val[i:i+INFER_BATCH]).to(device, non_blocking=True)
        recon = model(batch)
        mse   = torch.mean((batch - recon) ** 2, dim=1)
        val_errors.extend(mse.cpu().numpy())

val_errors = np.array(val_errors)

print("Threshold Sweep (percentiles on validation benign data):")
print(f"{'Percentile':>12} {'Threshold':>12} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 60)

for pct in [90, 93, 95, 97, 99]:
    thresh = np.percentile(val_errors, pct)
    preds  = (reconstruction_errors > thresh).astype(int)
    prec   = precision_score(y_test, preds, zero_division=0)
    rec    = recall_score(y_test, preds, zero_division=0)
    f1     = f1_score(y_test, preds, zero_division=0)
    print(f"{pct:>11}% {thresh:>12.6f} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f}")

In [ ]:
# Final classification report using Youden's J optimal threshold
y_pred = (reconstruction_errors > opt_thresh).astype(int)

print(f"Classification Report (Optimal Threshold = {opt_thresh:.6f}):")
print()
print(classification_report(y_test, y_pred, target_names=['Benign', 'Attack'], digits=4))
print(f"ROC AUC: {roc_auc:.4f}")

In [ ]:
# Confusion Matrix heatmap
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted Benign', 'Predicted Attack'],
    yticklabels=['Actual Benign', 'Actual Attack']
)
plt.title('Confusion Matrix — Autoencoder CICIDS2018')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  (Benign correct):  {tn:,}")
print(f"False Positives (Benign as Attack): {fp:,}")
print(f"False Negatives (Attack as Benign): {fn:,}")
print(f"True Positives  (Attack correct):   {tp:,}")
print(f"\nFalse Positive Rate: {fp/(fp+tn):.4f}")
print(f"Detection Rate:      {tp/(tp+fn):.4f}")

## 7. Latent Space Visualization

In [ ]:
# Extract latent embeddings for a subset of the test data
VIZ_SAMPLES = 5000
np.random.seed(SEED)

b_idx = np.random.choice(np.where(y_test == 0)[0],
                          min(VIZ_SAMPLES // 2, (y_test == 0).sum()), replace=False)
a_idx = np.random.choice(np.where(y_test == 1)[0],
                          min(VIZ_SAMPLES // 2, (y_test == 1).sum()), replace=False)
viz_idx = np.concatenate([b_idx, a_idx])

X_viz = X_test[viz_idx]
y_viz = y_test[viz_idx]

latent_reps = []
model.eval()
with torch.no_grad():
    for i in range(0, len(X_viz), 512):
        batch  = torch.FloatTensor(X_viz[i:i+512]).to(device)
        latent = model.encoder(batch)
        latent_reps.append(latent.cpu().numpy())

latent_reps = np.vstack(latent_reps)
print(f"Latent representations shape: {latent_reps.shape}")

In [ ]:
# PCA of latent space
pca = PCA(n_components=2, random_state=SEED)
latent_2d = pca.fit_transform(latent_reps)

plt.figure(figsize=(10, 7))
colors = {0: 'steelblue', 1: 'crimson'}
labels = {0: 'Benign', 1: 'Attack'}

for cls in [0, 1]:
    mask = y_viz == cls
    plt.scatter(latent_2d[mask, 0], latent_2d[mask, 1],
                c=colors[cls], label=labels[cls],
                alpha=0.4, s=15, edgecolors='none')

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('PCA of Autoencoder Latent Space — CICIDS2018')
plt.legend(markerscale=3)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Save Results to Google Drive

In [ ]:
# Compile and save results
results = {
    'dataset': 'CICIDS2018',
    'model': 'Autoencoder',
    'optimal_threshold': float(opt_thresh),
    'roc_auc': float(roc_auc),
    'accuracy':  float(accuracy_score(y_test, y_pred)),
    'precision': float(precision_score(y_test, y_pred, zero_division=0)),
    'recall':    float(recall_score(y_test, y_pred, zero_division=0)),
    'f1_score':  float(f1_score(y_test, y_pred, zero_division=0)),
    'false_positive_rate': float(fp / (fp + tn)),
    'detection_rate':      float(tp / (tp + fn)),
    'best_val_loss': float(best_val_loss),
    'input_dim':    input_dim,
    'encoding_dim': ENCODING_DIM,
    'batch_size':   BATCH_SIZE,
    'epochs_trained': len(train_losses),
    'device': str(device),
    'normal_samples': len(X_normal),
    'attack_samples': len(X_attack),
}

print("Final Results Summary:")
print("=" * 45)
for k, v in results.items():
    if isinstance(v, float):
        print(f"  {k:30s}: {v:.4f}")
    else:
        print(f"  {k:30s}: {v}")

with open(RESULTS_JSON_PATH, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Results saved to: {RESULTS_JSON_PATH}")
print(f"✅ Model saved to:   {MODEL_SAVE_PATH}")

In [ ]:
# Optional: Download the model weights directly from Colab
# (In addition to the Google Drive copy above)
from google.colab import files

download_model = False  # Set to True to trigger download

if download_model:
    files.download(MODEL_SAVE_PATH)
    files.download(RESULTS_JSON_PATH)
    print("Downloads started.")
else:
    print("Set download_model=True above to download files locally.")